In [8]:
!pip install -U langchain langchain-community langchain-openai langchain-chroma faiss-cpu sentence-transformers pypdf langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.5 MB/s eta 0:00:00


### ⚠️ Session Restart Required
After running the `pip install` above, you **must** restart the session for the imports to work. Click the button below or go to **Runtime -> Restart session**.

In [ ]:
import os
# This will restart the Colab runtime
os.kill(os.getpid(), 9)

## 1. Setup API Keys
You will need an API key for your LLM provider (e.g., OpenAI). Add your key to the Colab secrets manager as `OPENAI_API_KEY`.

In [1]:
import os
from google.colab import userdata

# Set keys for both providers
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except Exception:
    print("Note: OPENAI_API_KEY not found in secrets.")

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("✅ Google API Key set successfully.")
except Exception:
    print("❌ Action Required: Please add 'GOOGLE_API_KEY' to your Colab Secrets (key icon on the left) and toggle access.")

❌ Action Required: Please add 'GOOGLE_API_KEY' to your Colab Secrets (key icon on the left) and toggle access.


## 2. Document Loading and Vectorization
We will load documents, split them into chunks, and store them in a vector database.

In [2]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load a sample dataset (using a Wikipedia page as a corpus)
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
docs = loader.load()

# Split documents into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Use local HuggingFace embeddings instead of OpenAI to avoid quota issues
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

print(f"Successfully indexed {len(splits)} chunks into the vector store using local embeddings.")

/tmp/ipykernel_8592/4034482942.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
/tmp/ipykernel_8592/4034482942.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, cre

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully indexed 319 chunks into the vector store using local embeddings.


## 3. RAG Chain with Memory
Now we create a chain that retrieves relevant context and maintains conversation history.

In [5]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# Check if API Key is available before initializing
api_key = os.environ.get("GOOGLE_API_KEY")

if not api_key:
    print("❌ Error: GOOGLE_API_KEY not found.")
    print("Please add it to your Colab Secrets (key icon on the left) and restart the setup cell.")
else:
    # Initialize the LLM
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

    # 1. Prompt to contextualize the question
    contextualize_q_system_prompt = (
        "Given a chat history and the latest user question "
        "which might reference context in the chat history, "
        "formulate a standalone question which can be understood "
        "without the chat history."
    )
    contextualize_q_prompt = ChatPromptTemplate.from_messages([
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ])

    # 2. Prompt for answering the question
    qa_system_prompt = (
        "You are an assistant for question-answering tasks. "
        "Use the following pieces of retrieved context to answer "
        "the question. If you don't know the answer, just say that "
        "you don't know. Use three sentences maximum and keep the "
        "answer concise.\n\n"
        "{context}"
    )
    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", qa_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ])

    # Helper to format docs
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # Build the LCEL Chain
    retriever = vectorstore.as_retriever()

    # Step A: Create a standalone question generator
    contextualize_chain = contextualize_q_prompt | llm | StrOutputParser()

    # Step B: Create the RAG chain
    def get_context(input_dict):
        if input_dict.get("chat_history"):
            query = contextualize_chain.invoke(input_dict)
        else:
            query = input_dict["input"]
        docs = retriever.invoke(query)
        return format_docs(docs)

    rag_chain = (
        RunnablePassthrough.assign(context=get_context)
        | qa_prompt
        | llm
        | StrOutputParser()
    )

    # Example usage
    chat_history = []

    try:
        print("Question 1: What is AI?")
        ans1 = rag_chain.invoke({"input": "What is AI?", "chat_history": chat_history})
        print(f"Assistant: {ans1}")

        chat_history.extend([("human", "What is AI?"), ("assistant", ans1)])

        print("\nQuestion 2: What are its risks?")
        ans2 = rag_chain.invoke({"input": "What are its risks?", "chat_history": chat_history})
        print(f"Assistant: {ans2}")
    except Exception as e:
        print(f"An error occurred during execution: {e}")

❌ Error: GOOGLE_API_KEY not found.
Please add it to your Colab Secrets (key icon on the left) and restart the setup cell.
